<a href="https://colab.research.google.com/github/Godstouch/GNN-Student-Risk-Prediction-/blob/main/Feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.read_excel('/content/drive/MyDrive/Real school data (2).xlsx')


jhs_mask = df['level_prefix'] == 'JHS'
df.loc[jhs_mask, 'grade_number'] = df.loc[jhs_mask, 'grade_number'] + 6

# Clean jitter noise on grade levels
df['grade_number'] = df['grade_number'].round().astype('Int64')

print("Grade number distribution after fix:")
print(df['grade_number'].value_counts().sort_index())

#  Build weekly attendance rate (mean of Week1-14 attendance)
week_cols = [f'Week{i}_attendance' for i in range(1, 15)]
df['attendance_rate'] = df[week_cols].mean(axis=1)

# Consecutive absence proxy: count of weeks below 0.5 attendance
df['low_attendance_weeks'] = (df[week_cols] < 0.5).sum(axis=1)


df['academic_domain'] = (
    0.4 * df['Semester 1 average'] +
    0.4 * df['Semester 2 average'] +
    0.2 * df['Semester difference'].clip(lower=0)  # improvement rewarded
).clip(0, 100)

# Attendance domain: mean weekly attendance rate, scaled to 0-100
df['attendance_domain'] = (df['attendance_rate'] * 100).clip(0, 100)

# Socioeconomic domain: income level + family/child-labor flags
income_map = {'Low': 0, 'Average': 50, 'High': 100}
df['income_score'] = df['Household income level (standardized)'].map(income_map)
df['family_dropout_penalty'] = df['Family dropout history'].map({'Yes': -30, 'No': 0})
df['child_labor_penalty'] = df['Child labor involvement'].map({'Yes': -30, 'No': 0})
df['socioeconomic_domain'] = (
    df['income_score'] + df['family_dropout_penalty'] + df['child_labor_penalty']
).clip(0, 100)

# Engagement domain: extracurricular participation + relationship quality
relationship_map = {'Poor': 0, 'Average': 50, 'Good': 100}
df['teacher_rel_score'] = df['Teacher relationship quality'].map(relationship_map)
df['peer_rel_score'] = df['Peer relationship quality'].map(relationship_map)
df['has_extracurricular'] = df['Extra-curricular activities'].notna().astype(int) * 100
df['engagement_domain'] = (
    0.4 * df['teacher_rel_score'] +
    0.4 * df['peer_rel_score'] +
    0.2 * df['has_extracurricular']
).clip(0, 100)

# Accessibility domain: inverse of commute burden
df['accessibility_domain'] = (
    100
    - (df['long_commute_flag'] * 40)
    - (df['long_walk_flag'] * 40)
    - (df['Travel time to school (minutes)'].clip(0, 60) / 60 * 20)
).clip(0, 100)

#  Unsupervised risk tiering via k-means (replaces hand-weighted risk_score + quantile cutoffs)
domain_cols = ['academic_domain', 'attendance_domain', 'socioeconomic_domain',
               'engagement_domain', 'accessibility_domain']

# Impute missing values in the domain columns
for col in domain_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mean())

scaler = StandardScaler()
domain_scaled = scaler.fit_transform(df[domain_cols])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(domain_scaled)


centers_df = pd.DataFrame(kmeans.cluster_centers_, columns=domain_cols)


composite_risk = centers_df[['academic_domain', 'attendance_domain']].mean(axis=1)

risk_order = {cluster: label for cluster, label in
              zip(composite_risk.sort_values().index, ['High', 'Medium', 'Low'])}
df['dropout_risk'] = df['cluster'].map(risk_order)

print("Corrected class distribution:")
print(df['dropout_risk'].value_counts())

centers_df['risk_label'] = centers_df.index.map(risk_order)
print("\nCorrected cluster centers with labels:")
print(centers_df)


df.to_csv('Real_school_data_with_proxy_corrected.csv', index=False)



from sklearn.metrics import silhouette_score
sil_score = silhouette_score(domain_scaled, df['cluster'])
print(f"\nSilhouette score: {sil_score:.3f}  (>0.25 is reasonable for social-science data)")


#. labeled dataset for graph construction

df.to_csv('Real_school_data_with_proxy_corrected.csv', index=False)
print("\nSaved: Real_school_data_with_proxy_corrected.csv, shape:", df.shape)

Grade number distribution after fix:
grade_number
4    191
5    216
6    248
7    275
8     70
Name: count, dtype: Int64
Corrected class distribution:
dropout_risk
Low       403
Medium    347
High      250
Name: count, dtype: int64

Corrected cluster centers with labels:
   academic_domain  attendance_domain  socioeconomic_domain  \
0        -0.977146           0.382271             -0.111779   
1        -0.193640          -1.006271              0.195969   
2         0.961487           0.295087             -0.025322   

   engagement_domain  accessibility_domain risk_label  
0           0.385595              0.010168     Medium  
1          -0.908065             -0.249321       High  
2           0.231302              0.145910        Low  

Silhouette score: 0.187  (>0.25 is reasonable for social-science data)

Saved: Real_school_data_with_proxy_corrected.csv, shape: (1000, 54)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
